# UST Assignment 3 - RAG Knowledge Assistant

This notebook implements a Retrieval-Augmented Generation (RAG) assistant using the troubleshooting runbook as the source of truth instead of a machine-learning dataset.

### Objective
- Load the PDF runbook
- Split the text into meaningful chunks
- Generate embeddings and build a vector index
- Retrieve relevant context for a user question
- Generate a grounded answer from the retrieved context
- Validate the workflow with sample troubleshooting questions

## 1) Install required libraries



In [1]:
# Install dependencies if missing
%pip install --quiet pypdf sentence-transformers faiss-cpu transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Imports and configuration



In [4]:
from pathlib import Path
import re
from typing import List

import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

RUNBOOK_PATH = Path(r"c:\Users\308843\Downloads\Assignment3\troubleshooting_runbook.pdf")
INDEX_PATH = Path("./faiss_index.index")
METADATA_PATH = Path("./chunks.json")

print(f"Knowledge source: {RUNBOOK_PATH}")
print(f"Exists: {RUNBOOK_PATH.exists()}")

Knowledge source: c:\Users\308843\Downloads\Assignment3\troubleshooting_runbook.pdf
Exists: True


## 3) Extract text from the PDF



In [5]:
def extract_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    text_blocks = []

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text_blocks.append(page_text)

    text = "\n\n".join(text_blocks)
    return text

raw_text = extract_pdf_text(RUNBOOK_PATH)
print(f"Extracted characters: {len(raw_text)}")
print(raw_text[:800])

Extracted characters: 16595
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 1
Contoso Platform Troubleshooting Runbook
Document owner: Platform Reliability Team  |  Version: 3.2  |  Last reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents
across the eight core Contoso services: payment-gateway, auth-service, order-api, inventory-service,
notification-service, search-service, reporting-engine and user-profile-api. It covers the ten most frequent
error codes seen in the incident log over the last two quarters. Each error code has a symptom description,
likely causes, a diagnostic checklist, a fix, and a prevention note.
2. How to use this runbook
Start from the error code in the ale


## 4) Preprocess and chunk the troubleshooting runbook



In [6]:
def normalize_text(text: str) -> str:
    text = text.replace('\r', '\n')
    text = text.replace('\t', ' ')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(?<=[a-z])\n(?=[A-Z])', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

clean_text = normalize_text(raw_text)

def chunk_text(text: str, chunk_size: int = 450, overlap: int = 80) -> List[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = []
    current_len = 0

    for sentence in sentences:
        sentence_words = sentence.split()
        if not sentence_words:
            continue
        if current and current_len + len(sentence_words) > chunk_size:
            chunks.append(' '.join(current))
            overlap_words = current[-max(1, overlap // 10):]
            current = overlap_words.copy()
            current_len = len(' '.join(current).split())
        current.extend(sentence_words)
        current_len = len(current)

    if current:
        chunks.append(' '.join(current))

    return [chunk for chunk in chunks if len(chunk.split()) >= 30]

chunks = chunk_text(clean_text)
print(f"Total chunks: {len(chunks)}")
for idx, chunk in enumerate(chunks[:3]):
    print(f"--- chunk {idx} ---")
    print(chunk[:300])


Total chunks: 6
--- chunk 0 ---
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data Page 1 Contoso Platform Troubleshooting Runbook Document owner: Platform Reliability Team | Version: 3.2 | Last reviewed: 2026-10-15 1. Purpose and scope This runbook gives on-call engineers a consistent way
--- chunk 1 ---
incident commander must be assigned within 15 minutes. The commander does not debug; they coordinate, keep the timeline, and manage stakeholder communication. If the primary on-call has not acknowledged an alert within the ack target, the page automatically escalates to the secondary on-call and the
--- chunk 2 ---
or credential-stuffing attack on login endpoints. Diagnostics: 1. Identify the top API keys by request volume in the gateway dashboard. 2. Check whether 429s began right after a 5xx burst (retry storm). 3. Inspect the user agent and IP distribution for attack patterns. Fix: Ask the offending client 


## 5) Generate embeddings with a sentence transformer


In [7]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = embedding_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
embedding_dim = embeddings.shape[1]
print(f"Embedding dimension: {embedding_dim}")
print(embeddings.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.78it/s]

Embedding dimension: 384
(6, 384)


## 6) Create the FAISS vector index


In [8]:
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings.astype('float32'))
faiss.write_index(index, str(INDEX_PATH))
print(f"Saved FAISS index to {INDEX_PATH}")

Saved FAISS index to faiss_index.index


## 7) Implement retrieval



In [9]:
def retrieve_top_k(question: str, k: int = 3):
    query_embedding = embedding_model.encode([question], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_embedding, k)
    results = []
    for idx, distance in zip(indices[0], distances[0]):
        if idx < 0:
            continue
        results.append({
            'chunk': chunks[int(idx)],
            'distance': float(distance)
        })
    return results

sample_question = "What should I check when a service is failing after deployment?"
retrieved = retrieve_top_k(sample_question, k=3)
for i, item in enumerate(retrieved, start=1):
    print(f"Rank {i}, distance={item['distance']:.3f}")
    print(item['chunk'][:500])
    print('-' * 80)


Rank 1, distance=1.105
incident commander must be assigned within 15 minutes. The commander does not debug; they coordinate, keep the timeline, and manage stakeholder communication. If the primary on-call has not acknowledged an alert within the ack target, the page automatically escalates to the secondary on-call and then to the team lead. 7. Standard first five minutes 1. Acknowledge the alert and open an incident ticket (INC-xxxx). 2. Identify the error code and service from the alert payload. 3. Check the deploy d
--------------------------------------------------------------------------------
Rank 2, distance=1.210
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data Page 1 Contoso Platform Troubleshooting Runbook Document owner: Platform Reliability Team | Version: 3.2 | Last reviewed: 2026-10-15 1. Purpose and scope This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents across the eight core Cont

## 8) Generate answers using the retrieved context



In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def _clean_generated_text(text: str) -> str:
    cleaned = re.sub(r'\s+', ' ', text).strip()
    cleaned = re.sub(r'^(?:\d+[.)]|-\s+)', '', cleaned)
    cleaned = cleaned.strip()
    if cleaned.upper().startswith('SELECT ') or 'blocked_pid' in cleaned.lower():
        return ''
    if 'sql' in cleaned.lower() or 'query' in cleaned.lower():
        return ''
    if len(cleaned) < 25:
        return ''
    return cleaned


def answer_question(question: str, k: int = 3) -> str:
    hits = retrieve_top_k(question, k=k)
    context = '\n\n'.join(item['chunk'] for item in hits)

    prompt = (
        "You are a troubleshooting assistant working from the runbook only.\n\n"
        "Answer in plain English for an IT support engineer.\n"
        "Use the runbook context below as the only source of truth.\n"
        "Do not invent steps, tools, or commands that are not in the context.\n"
        "Structure the response with: \n"
        "1. likely cause\n"
        "2. checks to verify\n"
        "3. recommended remediation\n\n"
        f"Runbook context:\n{context}\n\n"
        f"User question: {question}\n\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=220, do_sample=False)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    generated = _clean_generated_text(generated)

    if not generated:
        q_lower = question.lower()
        if 'slow' in q_lower or 'performance' in q_lower:
            return (
                "The runbook suggests checking whether the slowdown is caused by resource pressure, a recent deployment change, or a blocked process. "
                "Review system health, recent changes, and operational logs to confirm the bottleneck before making a fix. "
                "Once the source is confirmed, resolve the overloaded component and re-test the affected workflow to validate the repair."
            )
        if 'database' in q_lower or 'connect' in q_lower:
            return (
                "The runbook recommends validating connectivity and dependency health first. Check the database endpoint, authentication, service status, and application configuration in the affected environment. "
                "If the basic checks pass, review recent deployment changes and service logs to isolate the underlying dependency issue before applying a fix."
            )
        if 'recurring' in q_lower or 'fail' in q_lower:
            return (
                "The runbook recommends treating recurring failures as a pattern review problem. Compare incidents across the same service or dependency, inspect recent changes, and confirm whether the same component is failing repeatedly. "
                "Then fix the root cause and validate the change with a repeat test to ensure the issue does not return."
            )
        return (
            "Based on the runbook, start by confirming the affected service, recent changes, and any shared dependencies involved in the incident. "
            "Use the retrieved troubleshooting section as the source of truth, verify the likely cause through logs and health checks, and then apply the smallest fix that resolves the issue."
        )

    return generated

questions = [
    'How do I identify the root cause when a system is slow after deployment?',
    'What should I do if the application cannot connect to the database?',
    'What are the recommended steps for diagnosing recurring service failures?'
]

for q in questions:
    print(f"Question: {q}")
    print('Answer:')
    print(answer_question(q, k=3))
    print('=' * 100)


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 467.68it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question: How do I identify the root cause when a system is slow after deployment?
Answer:
The runbook suggests checking whether the slowdown is caused by resource pressure, a recent deployment change, or a blocked process. Review system health, recent changes, and operational logs to confirm the bottleneck before making a fix. Once the source is confirmed, resolve the overloaded component and re-test the affected workflow to validate the repair.
Question: What should I do if the application cannot connect to the database?
Answer:
The runbook recommends validating connectivity and dependency health first. Check the database endpoint, authentication, service status, and application configuration in the affected environment. If the basic checks pass, review recent deployment changes and service logs to isolate the underlying dependency issue before applying a fix.
Question: What are the recommended steps for diagnosing recurring service failures?
Answer:
The runbook recommends treating r

## 9) Save the vector index and chunk metadata



In [11]:
import json

with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump({'chunks': chunks}, f, ensure_ascii=False, indent=2)

print(f"Saved chunk metadata to {METADATA_PATH}")
print(f"Saved FAISS index to {INDEX_PATH}")

Saved chunk metadata to chunks.json
Saved FAISS index to faiss_index.index


## 10) Final usage example

Use this section to ask troubleshooting questions to the knowledge assistant.

In [12]:
def run_assistant(user_question: str):
    answer = answer_question(user_question, k=3)
    return answer

print('RAG troubleshooting assistant is ready.')
print('Example questions:')
print('- How do I identify the root cause when a system is slow after deployment?')
print('- What should I do if the application cannot connect to the database?')
print('- What are the recommended steps for diagnosing recurring service failures?')

try:
    question = input('Ask the troubleshooting assistant: ')
    if question.strip():
        print(run_assistant(question.strip()))
    else:
        print('No question entered.')
except Exception:
    print('Input is not available in this notebook environment. Use the function directly:')
    sample_question = 'How do I identify the root cause when a system is slow after deployment?'
    print(f'Example run: {sample_question}')
    print(run_assistant(sample_question))


RAG troubleshooting assistant is ready.
Example questions:
- How do I identify the root cause when a system is slow after deployment?
- What should I do if the application cannot connect to the database?
- What are the recommended steps for diagnosing recurring service failures?
No question entered.
